In [1]:
from aocd import get_data
data_real = get_data(day=8, year=2024)
data_real[:40]

'....1.y.D...Y..........w....m...........'

In [2]:
data_test = """............
........0...
.....0......
.......0....
....0.......
......A.....
............
............
........A...
.........A..
............
............"""

# Part 1
You find yourselves on the roof of a top-secret Easter Bunny installation.

While The Historians do their thing, you take a look at the familiar huge antenna. Much to your surprise, it seems to have been reconfigured to emit a signal that makes people 0.1% more likely to buy Easter Bunny brand Imitation Mediocre Chocolate as a Christmas gift! Unthinkable!

Scanning across the city, you find that there are actually many such antennas. Each antenna is tuned to a specific frequency indicated by a single lowercase letter, uppercase letter, or digit. You create a map (your puzzle input) of these antennas. For example:

```
............
........0...
.....0......
.......0....
....0.......
......A.....
............
............
........A...
.........A..
............
............
```

The signal only applies its nefarious effect at specific antinodes based on the resonant frequencies of the antennas. In particular, an antinode occurs at any point that is perfectly in line with two antennas of the same frequency - but only when one of the antennas is twice as far away as the other. This means that for any pair of antennas with the same frequency, there are two antinodes, one on either side of them.

So, for these two antennas with frequency a, they create the two antinodes marked with #:
```
..........
...#......
..........
....a.....
..........
.....a....
..........
......#...
..........
..........
```
Adding a third antenna with the same frequency creates several more antinodes. It would ideally add four antinodes, but two are off the right side of the map, so instead it adds only two:
```
..........
...#......
#.........
....a.....
........a.
.....a....
..#.......
......#...
..........
..........
```
Antennas with different frequencies don't create antinodes; A and a count as different frequencies. However, antinodes can occur at locations that contain antennas. In this diagram, the lone antenna with frequency capital A creates no antinodes but has a lowercase-a-frequency antinode at its location:
```
..........
...#......
#.........
....a.....
........a.
.....a....
..#.......
......A...
..........
..........
```
The first example has antennas with two different frequencies, so the antinodes they create look like this, plus an antinode overlapping the topmost A-frequency antenna:
```
......#....#
...#....0...
....#0....#.
..#....0....
....0....#..
.#....A.....
...#........
#......#....
........A...
.........A..
..........#.
..........#.
```
Because the topmost A-frequency antenna overlaps with a 0-frequency antinode, there are 14 total unique locations that contain an antinode within the bounds of the map.

Calculate the impact of the signal. How many unique locations within the bounds of the map contain an antinode?

Plan to solve this challenge:
* Parse all nodes as (type, x, y)
* For each type: get all node pairs and create antinode coords
* Remove antinodes outside of map
* done

In [13]:
data = data_test

nodes = [(col,x,y) for y,row in enumerate(data.splitlines()) for x,col in enumerate(row) if col != "."]
nodes[:5]

[('0', 8, 1), ('0', 5, 2), ('0', 7, 3), ('0', 4, 4), ('A', 6, 5)]

In [32]:
node_types = list(set([n[0] for n in nodes]))
node_types

['A', '0']

In [33]:
from itertools import product

def get_pairs(node_type, nodes):
    nodes_ = [n for n in nodes if n[0]==node_type]
    return [p for p in list(product(nodes_, repeat=2)) if p[0] != p[1]]
get_pairs("A", nodes)    

[(('A', 6, 5), ('A', 8, 8)),
 (('A', 6, 5), ('A', 9, 9)),
 (('A', 8, 8), ('A', 6, 5)),
 (('A', 8, 8), ('A', 9, 9)),
 (('A', 9, 9), ('A', 6, 5)),
 (('A', 9, 9), ('A', 8, 8))]

In [42]:
def get_an(pair):
    n1, n2 = pair
    offset = n2[1] - n1[1], n2[2] - n1[2]
    an1 = "#", n1[1] - offset[0], n1[2] - offset[1]
    an2 = "#", n2[1] + offset[0], n2[2] + offset[1]
    return an1, an2
# n1, n2, offset, an1, an2
get_an(get_pairs("A", nodes)[0])

(('#', 4, 2), ('#', 10, 11))

In [49]:
anodes = [get_an(pair) for nt in node_types for pair in get_pairs(nt, nodes)]
anodes[:5]

[(('#', 4, 2), ('#', 10, 11)),
 (('#', 3, 1), ('#', 12, 13)),
 (('#', 10, 11), ('#', 4, 2)),
 (('#', 7, 7), ('#', 10, 10)),
 (('#', 12, 13), ('#', 3, 1))]

let's write a function to visualize the data

In [108]:
def visualize_an(anodes, data):
    m=[[x for x in row] for row in data.splitlines()]
    height, width = len(m), len(m[0])
    for t,x,y in anodes:
        if -1<x<width and -1<y<height:
            if m[y][x]==".":
                m[y][x] = t
    print("\n".join(["".join(row) for row in m]))
visualize_an(anodes, data)

......#..#..
........0...
..#..0......
.......0#..#
....0.......
..#...A.....
......#.....
#..#...#....
..#.....A#..
..#....#.A#.
#.###.#..#..
#...#......#


In [74]:
height,width = len(data.splitlines()), len(data.splitlines()[0])
height, width

(12, 12)

ok, let's put it all together

In [82]:
data = data_test

height,width = len(data.splitlines()), len(data.splitlines()[0])
nodes = [(col,x,y) for y,row in enumerate(data.splitlines()) for x,col in enumerate(row) if col != "."]
node_types = list(set([n[0] for n in nodes]))
anodes = [an for nt in node_types for pair in get_pairs(nt, nodes) for an in get_an(pair)]
valid_anodes = list(set([(t,x,y) for t,x,y in anodes if 0 <= x < width and 0 <= y < height]))
len(valid_anodes), valid_anodes[:5]

(14, [('#', 6, 5), ('#', 3, 6), ('#', 10, 2), ('#', 9, 4), ('#', 0, 7)])

In [83]:
data = data_real

height,width = len(data.splitlines()), len(data.splitlines()[0])
nodes = [(col,x,y) for y,row in enumerate(data.splitlines()) for x,col in enumerate(row) if col != "."]
node_types = list(set([n[0] for n in nodes]))
anodes = [an for nt in node_types for pair in get_pairs(nt, nodes) for an in get_an(pair)]
valid_anodes = list(set([(t,x,y) for t,x,y in anodes if 0 <= x < width and 0 <= y < height]))
len(valid_anodes), valid_anodes[:5]

(367,
 [('#', 32, 18), ('#', 48, 44), ('#', 32, 27), ('#', 29, 40), ('#', 44, 1)])

# Part 2
Watching over your shoulder as you work, one of The Historians asks if you took the effects of resonant harmonics into your calculations.

Whoops!

After updating your model, it turns out that an antinode occurs at any grid position exactly in line with at least two antennas of the same frequency, regardless of distance. This means that some of the new antinodes will occur at the position of each antenna (unless that antenna is the only one of its frequency).

So, these three T-frequency antennas now create many antinodes:
```
T....#....
...T......
.T....#...
.........#
..#.......
..........
...#......
..........
....#.....
..........
```
In fact, the three T-frequency antennas are all exactly in line with two antennas, so they are all also antinodes! This brings the total number of antinodes in the above example to 9.

The original example now has 34 antinodes, including the antinodes that appear on every antenna:

```
##....#....#
.#.#....0...
..#.#0....#.
..##...0....
....0....#..
.#...#A....#
...#..#.....
#....#.#....
..#.....A...
....#....A..
.#........#.
...#......##
```
Calculate the impact of the signal using this updated model. How many unique locations within the bounds of the map contain an antinode?

Plan:
* OK, this means we need to update the get_an function to repeat putting antinodes until the boundary of the map is reached

In [97]:
data = data_test

height,width = len(data.splitlines()), len(data.splitlines()[0])
nodes = [(col,x,y) for y,row in enumerate(data.splitlines()) for x,col in enumerate(row) if col != "."]
node_types = list(set([n[0] for n in nodes]))
visualize_an([], data)
height,width

............
........0...
.....0......
.......0....
....0.......
......A.....
............
............
........A...
.........A..
............
............


(12, 12)

In [103]:
# def get_an(pair, height, width):
# n1, n2, offset, an1, an2
# get_an(get_pairs("A", nodes)[0], height, width)

pair = get_pairs("A", nodes)[-1]
n1, n2 = pair
offset = n2[1] - n1[1], n2[2] - n1[2]
an = []
x, y = n1[1], n1[2]
while (0<= x < width and 0 <= y < height):
    an.append(("#", x, y))
    x-=offset[0]
    y-=offset[1]
x, y = n2[1], n2[2]
while (0<= x < width and 0 <= y < height):
    an.append(("#", x, y))
    x+=offset[0]
    y+=offset[1]
an
# visualize_an(an, data)

[('#', 9, 9),
 ('#', 10, 10),
 ('#', 11, 11),
 ('#', 8, 8),
 ('#', 7, 7),
 ('#', 6, 6),
 ('#', 5, 5),
 ('#', 4, 4),
 ('#', 3, 3),
 ('#', 2, 2),
 ('#', 1, 1),
 ('#', 0, 0)]

In [109]:
visualize_an(an, data)

#...........
.#......0...
..#..0......
...#...0....
....0.......
.....#A.....
......#.....
.......#....
........A...
.........A..
..........#.
...........#


In [111]:
def get_an(pair, height, width):
    n1, n2 = pair
    offset = n2[1] - n1[1], n2[2] - n1[2]
    an = []
    x, y = n1[1], n1[2]
    while (0<= x < width and 0 <= y < height):
        an.append(("#", x, y))
        x-=offset[0]
        y-=offset[1]
    x, y = n2[1], n2[2]
    while (0<= x < width and 0 <= y < height):
        an.append(("#", x, y))
        x+=offset[0]
        y+=offset[1]
    return an
get_an(get_pairs("A", nodes)[0], height, width)

[('#', 6, 5), ('#', 4, 2), ('#', 8, 8), ('#', 10, 11)]

In [113]:
data = data_test

height,width = len(data.splitlines()), len(data.splitlines()[0])
nodes = [(col,x,y) for y,row in enumerate(data.splitlines()) for x,col in enumerate(row) if col != "."]
node_types = list(set([n[0] for n in nodes]))
anodes = [an for nt in node_types for pair in get_pairs(nt, nodes) for an in get_an(pair, height, width)]
valid_anodes = list(set([(t,x,y) for t,x,y in anodes if 0 <= x < width and 0 <= y < height]))
len(valid_anodes), valid_anodes[:5]

(34, [('#', 6, 5), ('#', 3, 6), ('#', 10, 2), ('#', 9, 4), ('#', 8, 8)])

In [114]:
data = data_real

height,width = len(data.splitlines()), len(data.splitlines()[0])
nodes = [(col,x,y) for y,row in enumerate(data.splitlines()) for x,col in enumerate(row) if col != "."]
node_types = list(set([n[0] for n in nodes]))
anodes = [an for nt in node_types for pair in get_pairs(nt, nodes) for an in get_an(pair, height, width)]
valid_anodes = list(set([(t,x,y) for t,x,y in anodes if 0 <= x < width and 0 <= y < height]))
len(valid_anodes), valid_anodes[:5]

(1285,
 [('#', 32, 27), ('#', 44, 1), ('#', 33, 28), ('#', 25, 24), ('#', 45, 2)])